In [1]:
from webspirit.tools.checktype import CheckType, HyperLink, StrPath

In [2]:
import json

@CheckType('notebook')
def links_from_cell(notebook: StrPath, index: int = 0) -> list[HyperLink]:
    with notebook.open('r', encoding='utf-8') as f:
        notebook = json.load(f)

    cell: dict = notebook['cells'][index]

    return [
        HyperLink(link.removesuffix('\n').removesuffix('  ')) for link in cell['source']
    ]   if cell['cell_type'] in ('markdown', 'raw') else ['']

filename: str = 'pictures.ipynb'
index: int = 1

links: list[HyperLink] = links_from_cell(filename, index)

print(links)

2025-08-30 15:03:25  webspirit.config.logger  DEBUG    Change 'pictures.ipynb' of type <class 'str'> to type <class 'webspirit.tools.checktype.StrPath'>


[HyperLink('https://www.youtube.com/watch?v=Ik5Nh94v7EQ'), HyperLink('https://www.youtube.com/watch?v=09R8_2nJtjg')]


In [3]:
DEFAULT: int = 0     # 120x90
MEDIUM: int = 1      # 320x180
HIGH: int = 2        # 480x360
STANDARD: int = 3    # 640x480
MAXIMUM: int = 4     # Jusqu'à 1280x720

In [4]:
img_links: list[dict[str, str]] = [
    {
        'id' : link.id,
        DEFAULT : HyperLink(f"https://img.youtube.com/vi/{link.id}/0.jpg"),
        MEDIUM : HyperLink(f"https://img.youtube.com/vi/{link.id}/mqdefault.jpg"),
        HIGH : HyperLink(f"https://img.youtube.com/vi/{link.id}/hqdefault.jpg"),
        STANDARD : HyperLink(f"https://img.youtube.com/vi/{link.id}/sddefault.jpg"),
        MAXIMUM : HyperLink(f"https://img.youtube.com/vi/{link.id}/maxresdefault.jpg")
    }
    for link in links
]

print(img_links)

[{'id': 'Ik5Nh94v7EQ', 0: HyperLink('https://img.youtube.com/vi/Ik5Nh94v7EQ/0.jpg'), 1: HyperLink('https://img.youtube.com/vi/Ik5Nh94v7EQ/mqdefault.jpg'), 2: HyperLink('https://img.youtube.com/vi/Ik5Nh94v7EQ/hqdefault.jpg'), 3: HyperLink('https://img.youtube.com/vi/Ik5Nh94v7EQ/sddefault.jpg'), 4: HyperLink('https://img.youtube.com/vi/Ik5Nh94v7EQ/maxresdefault.jpg')}, {'id': '09R8_2nJtjg', 0: HyperLink('https://img.youtube.com/vi/09R8_2nJtjg/0.jpg'), 1: HyperLink('https://img.youtube.com/vi/09R8_2nJtjg/mqdefault.jpg'), 2: HyperLink('https://img.youtube.com/vi/09R8_2nJtjg/hqdefault.jpg'), 3: HyperLink('https://img.youtube.com/vi/09R8_2nJtjg/sddefault.jpg'), 4: HyperLink('https://img.youtube.com/vi/09R8_2nJtjg/maxresdefault.jpg')}]


In [ ]:
from http.client import HTTPResponse

from urllib.request import urlopen
from urllib.parse import urljoin

from requests import get, Response

from bs4 import BeautifulSoup
from bs4.element import Tag

from PIL import Image, ImageDraw, ImageFont

from os.path import split, splitext, join
import os

from config.constants import DIR_PICTURE
from config.logger import log, INFO, ERROR
from profiles import Profile

from pathlib import Path

from typing import Optional

import threading

import shutil # to compress data in a .zip file

import re

# Define some function to 'download' slash command
async def load_page(url: str) -> BeautifulSoup:
    """Download html content with the url.

    Args:
        url (str): The url of a website.

    Returns:
        BeautifulSoup: Return a instance of BeautifulSoup.
    """
    try:
        response: HTTPResponse = urlopen(url)
        page: bytes = response.read()

    except Exception as e:
        log(str(e), ERROR)
        raise e

    return BeautifulSoup(page, 'html.parser')

async def extract_lien_from_balise(list_balises: list[Tag]) -> list[str]:
    """Extract different url contains in a list of balises with a 'src' or a 'data-src' attribute.

    Args:
        list_balises (list[Tag]): The list of balises.

    Returns:
        list[str]: Returns url contains in tags.
    """
    return [str(balise_image['src']) if str(balise_image['src']).startswith('https://') else str(balise_image['data-src']) for balise_image in list_balises]

async def load_balise_picture(page: BeautifulSoup) -> list[str]:
    """Find all 'img' balise in a page, and extract the content with the tag 'src'.

    Args:
        page (BeautifulSoup): The page.

    Returns:
        list[str]: Returns different url in the page.
    """
    balise_image: list[Tag] = list(page.find_all('img', src=True))
    liens = await extract_lien_from_balise(balise_image)
    return liens

async def formate_lien_picture(liste: list[str], url: str) -> list[str]:
    """Formate a list of url with a root url and specifics conditions.

    Args:
        liste (list[str]): The list of url.
        url (str): The root url.

    Returns:
        list[str]: Returns a list of url which can download something.
    """
    hyperliens: list[str] = [urljoin(url, L.replace(".\\","")).replace("\\","/") for L in list(set(liste))]
    return list(set(hyperliens))

def load_picture(url: str, filter: list[str], N: int = 0):
    """Download one picture with the url.

    Args:
        url (str): The url of the ressource.
        filter (list[str]): A selection of specific extensions: ['.png', '.jpg', ...]
        N (int): A number to make defaults pictures names.
    """
    global names

    name, ext = splitext(split(url)[1])

    surplus: int = ext.rfind("?")
    if surplus != -1: ext = ext[:surplus]
    if name in names or not re.match(r'^[\w,\s-]+\.[A-Za-z]{3}$', name): name: str = f"picture_{N}"
    if '.ashx' in ext: ext: str = '.png' # rename specific extensions in '.png'
    if not ext: ext = '.png'
    names.add(name)

    if ((filter[0] == '*' and ext) or ext in filter) and ext != '.svg':
        with open(name+ext, 'ab') as f:
            result: HTTPResponse = urlopen(url)
            image: bytes = result.read()
            f.write(image)

        log(f"[ACTION] Download {name+ext} at {url}", INFO)

async def load_url(interaction: Interaction, url: str, filter: list[str], directory: Path, N_url: int, send: bool = True):
    """Download pictures which is contains in a website url.

    Args:
        interaction (Interaction): The Discord interaction.
        url (str): The url of the website.
        filter (list[str]): A selection of specific format: ['.png', '.jpg', ...]
        directory (Path): The directory where pictures is saved.
        N_url (int): A number to name folder for this url.
        send (bool, optional): A bool to indicate if it send you files (to use if you want a .zip file, and don't want to see all differently files). Defaults to True.
    """
    
    if is_greater:
        await interaction.channel.send(f"The download at {url} started.")
    else:
        await interaction.response.send_message(f"The download at {url} started.")

    folder_directory: Path = directory / f"download_{N_url}"
    folder_directory.mkdir(exist_ok=True)
    os.chdir(folder_directory)

    balise_page: BeautifulSoup = await load_page(url)

    if not balise_page:
        await interaction.channel.send(f"An error was occurred. Stopping download.")
        return

    balise_picture: list[str] = await load_balise_picture(balise_page)
    hyperlien: list[str] = await formate_lien_picture(balise_picture, url)

    fils: list[threading.Thread] = []
    N: int = 0
    for lien in hyperlien:
        fil = threading.Thread(target=load_picture, args=(lien, filter, N))
        fils.append(fil)
        fil.start()
        N += 1

    for thread in [thread for thread in threading.enumerate() if thread in fils]:
        thread.join()

    while any(thread.is_alive() for thread in threading.enumerate() if thread in fils):
        pass

    names_pictures: list[str] = os.listdir()
    log(f"Download {N} pictures in {directory}", INFO)

    if send:
        for file in [File(name) for name in names_pictures]:
            await interaction.channel.send(file=file)
            log(f"[ACTION] Download {file.filename} at Discord in {interaction.channel}")

async def load_pictures(interaction: Interaction, url: list[str], filter: list[str], directory: Path, zip_directory: str, zip: bool = False, send: bool = True):
    """Download pictures which is contains in differently websites.

    Args:
        interaction (Interaction): The Discord interaction.
        url (list[str]):  A list of url of websites.
        filter (list[str]): A selection of specific format: ['.png', '.jpg', ...]
        directory (Path): The directory where pictures is saved.
        zip_directory (str): The directory where the file .zip is saved.
        zip (bool, optional): A bool to know if it send you a .zip file of all downloading. Defaults to False.
        send (bool, optional): A bool to indicate if it send you files (to use if you want a .zip file, and don't want to see all differently files). Defaults to True.
    """
    global names, is_greater
    nbr_url: int = len(url)
    names, is_greater = set(), nbr_url >= 2

    os.makedirs(directory, exist_ok=True) ; os.chdir(directory)

    if is_greater:
        await interaction.response.send_message(f"The download of {nbr_url} url started.")
    
    for N_url, lien in enumerate(url):
        log(f"[COMMAND] /download url={lien} filter={filter} directory={directory} zip_directory={zip_directory} zip={zip} send={send}", INFO)
        await load_url(interaction, lien, filter, directory, N_url, send)

    os.chdir(directory)

    if zip:
        os.chdir(zip_directory)
        shutil.make_archive('pictures', format='zip', root_dir='picture')
        log(f"[ACTION] Make 'pictures.zip' file in {zip_directory}", INFO)
        await interaction.channel.send(file=File('pictures.zip')) 
        log(f"[ACTION] Send 'pictures.zip' at Discord in {interaction.channel}", INFO)
        os.remove('pictures.zip')
        log(f"[ACTION] Delete 'pictures.zip' file in {zip_directory}", INFO)
        os.chdir(directory)

    N_del: int = 0
    for file in [folder for folder in os.listdir() if os.path.isdir(directory / folder)]:
        folder_directory: Path = directory / file
        os.chdir(folder_directory)
        names_pictures: list[str] = os.listdir()

        for name in names_pictures:
            os.remove(name) ; N_del += 1
            log(f"[ACTION] Delete {name} in {folder_directory}", INFO)
    
        os.chdir(directory) ; os.rmdir(file)

    log(f"Deleted {N_del} pictures in {directory}", INFO)
    await interaction.channel.send(f"The download of {N_del} pictures finished perfectly.")

# Define some function to 'extract' slash command
def save_profile_picture(url: str, path: Path):
    """Save a picture with its url.

    Args:
        url (str): The url of the picture.
        path (Path): The path where the picture will save.
    """
    directory: Path = Path(os.path.dirname(path))
    name: str = os.path.basename(path)

    os.makedirs(directory, exist_ok=True)
    os.chdir(directory)

    with path.open('ab') as f:
        result: Response = get(url)
        image: bytes = result.content

        f.write(image)

        log(f"[ACTION] Download {name} at {url}", INFO)

    log(f"Save {name} in {directory}", INFO)


In [5]:
from IPython.display import display, HTML

html = "<div style='display: flex; flex-wrap: wrap; gap: 10px;'>"
for url in img_links:
    html += f"<img src='{url}' style='width: 150px; height: auto;'>"
html += "</div>"

display(HTML(html))